# HL-mesh lab

Hexagonal L-meshes (HL-meshes) from the planar hexagonal meshes of PH-CPF (Pluta et al. 2021)
in `data/ph_results/`. Spheres on the mesh vertices, oriented planes on the hexagons.

- `hl_mesh_computation(name, tau)`: sphere radius = τ × smallest curvature-sphere radius,
  optimisation (oriented contact, strip and face fairness, proximity to the reference
  surface), `improve_contact()`, export to `notebooks/out/lmesh/HL_results/<mesh>/exp_<τ>/`.
- `hl_mesh_simple(name, tau)`: the same without a reference surface.
- Batch sweep: every mesh at several τ, with LaTeX tables and images.

Choose the mesh and τ in the last lines of the setup cell.

**Viewer:** `ws.init()` starts a kayviz viewer and prints its URL; open it in a browser. To share one viewer between several notebooks, run `python -m kayviz` in a terminal first.

## Python paths and configurations

data path: `data/ph_results`

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("",):  # project root: LMesh/, Lconjugacy/ (hanan, kayviz are installed)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", PROJECT_ROOT)

from LMesh.lmesh_serializers import lmesh_visualize
from kayviz.webscope import script as ws
import pymeshlab
from LMesh.L_mesh import Lmesh
from hanan.geometry import Mesh
from hanan.optimization.optimizer import Optimizer
from hanan.geometry.utils import (
    read_obj,
    read_off,
    normalize_vertices,
    compute_cone_axes,
    conical_mesh_smallest_curvature_sphere
)
import numpy as np
import igl

from hanan.optimization.optimizer import Optimizer
from LMesh.Optimization.Or_contactFree import Or_contactFree
from LMesh.Optimization.HLmesh_reg_strips import HLmesh_reg_strips
from LMesh.Optimization.HLmesh_reg_hex_faces import HLmesh_reg_hex_faces
from LMesh.Optimization.Tangential_Pos import TDPositive
from LMesh.Optimization.Tangential_Dist import Tangential_Dist
from LMesh.Optimization.ContactPointsProximity import ContactPointsProximity
from LMesh.Optimization.ContactPointsGlide import ContactPointsGlide

DATA     = os.path.join(PROJECT_ROOT, "data", "ph_results")
OUT_PATH = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "HL_results")

mesh_names = [
    "aspiralBdef.out_d_p",
    #"ph_input_cmcOptimized.out_d_p"
    #"ph_input_nautilus.out_d_p"
    "ph_input_nautilus_ex2.out_d_p"
    #"ph_input_nautilus_channel.out_d_p"
    #"ph_input_seashell_ex5.out_d_p"
    #"ph_input_seashell_ex6.out_d_p"
    #"ph_input_spiral_def.out_d_p"
    #"ph_input_spiral_defM.out_d_p"
    #"ph_input_testSimple.out_d_p"
]

def setup_optimizer(lmesh, Vr=None, Fr=None):
    opt = Optimizer()

    weights = {
            "Or_Contact": 1.0, 
            "TD": 0,
            "TDpos": 0,
            "HLmesh_Reg_Strips": 5e-3,
            "HLmesh_Reg_Hex_Faces": 5e-3, 
            "contactProximity": 1e-3,
            "contactGliding": 1e-4,
            "npf_fairness": 0    
            }

    cf  = lmesh.top.vertices[:, :3]
    rf  = lmesh.top.vertices[:, 3]
    npf = lmesh.PF[:, :3]
    hpf = lmesh.PF[:, 3]

    # npf/hpf are per-FACE variables (one oriented plane per L-mesh face),
    # so fairness must use face->face adjacency.  A vertex adjacency list
    # would index past the end of the face array.
    plane_adj = lmesh.top.face_face_adjacency_list()
    si_sj_e = lmesh.top.edge_vertices()  # edge->(start_vertex, end_vertex) pairs

    opt.add_variable("cv", cf.flatten())
    opt.add_variable("rv", rf)
    opt.add_variable("npf", npf.flatten())
    opt.add_variable("hpf", hpf.flatten())

    # Oriented contact term
    Or_contact = Or_contactFree()
    Or_contact.name = "or_contact"
    opt.add_objective_term(
        Or_contact,
        args=(lmesh.top,),
        w=weights["Or_Contact"],
        ce=1,
    )

    # Regularity of strips
    HLmesh_Reg_Strips = HLmesh_reg_strips()
    HLmesh_Reg_Strips.name = "HLmesh_Reg_Strips"
    opt.add_objective_term(
        HLmesh_Reg_Strips,
        args=(lmesh.top,),
        w=weights["HLmesh_Reg_Strips"],
        ce=1,
    )

    Tangential_Dist_term = Tangential_Dist()
    Tangential_Dist_term.name = "tangential_dist"
    opt.add_objective_term(
        Tangential_Dist_term,
        args=(si_sj_e, ),
        w=weights["TD"],
        ce=1,
    )

    Tangential_Dist_pos_term = Tangential_Dist()
    Tangential_Dist_pos_term.name = "tangential_dist_pos"
    opt.add_objective_term(
        Tangential_Dist_pos_term,
        args=(si_sj_e, ),
        w=weights["TDpos"],
        ce=1,
    )


    # Regularity of hex faces
    HLmesh_Reg_Hex_Faces = HLmesh_reg_hex_faces()
    HLmesh_Reg_Hex_Faces.name = "HLmesh_Reg_Hex_Faces"
    opt.add_objective_term(
        HLmesh_Reg_Hex_Faces,
        args=(lmesh.top,),
        w=weights["HLmesh_Reg_Hex_Faces"],
        ce=1,
    )

    if Vr is not None and Fr is not None:

        ContactPointsProximity_term = ContactPointsProximity()
        ContactPointsProximity_term.name = "contactProximity"
        opt.add_objective_term(
            ContactPointsProximity_term,
            args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
            w=weights["contactProximity"],
            ce=1,
        )
            
        ContactPointsGlide_term = ContactPointsGlide()
        ContactPointsGlide_term.name = "contactGliding"
        opt.add_objective_term(
            ContactPointsGlide_term,
            args=(lmesh.top.vertex_face_adjacency_list(), Vr, Fr),
            w=weights["contactGliding"],
            ce=1,
        )

    # Normal vector fairness
    opt.set_fairness("npf", plane_adj, 3, 0.1, 10, weights["npf_fairness"])
    opt.unitize_variable("npf", 3, 10)
    #opt.prev_iteration_regularizer(w=1e-4)
    opt.initialize_optimizer(verbose=True, adaptive_mu=False)


    return opt

def ref_surface_mesh(V, F):

  
    # Refine mesh for proximity 
    m  = pymeshlab.Mesh(vertex_matrix=V, face_matrix=F)
    ms = pymeshlab.MeshSet()
    ms.add_mesh(m)

    ms.apply_filter("meshing_isotropic_explicit_remeshing", 
                    targetlen=pymeshlab.PercentageValue(1.0),  # 1% of bbox diagonal
                    iterations=5)

    ms.apply_filter("apply_coord_taubin_smoothing", lambda_=0.5, mu=-0.53, stepsmoothnum=10)

    # Get the current mesh object
    mesh = ms.current_mesh()

    # Extract as numpy arrays
    Vr = mesh.vertex_matrix()       # (N, 3) float64 — vertex positions
    Fr = mesh.face_matrix()

    return Vr, Fr   

def hl_mesh_from_tri(name, tau):
    output_path = os.path.join(OUT_PATH, name)
    
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    print("Reading mesh:", name)

    V, F = read_obj(os.path.join(DATA, name + ".obj"))
    V = normalize_vertices(V, 2)

    F = [f for f in F if len(f) == 3]  # Filter out non-triangular faces

    F = np.array(F, dtype=np.int32)
    V = np.array(V, dtype=np.float64)

    ws.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()

    basemesh.make_mesh(V, F)

    _, _, k1, k2, _ = igl.principal_curvature(V, F)
    min_curvature = np.maximum( np.abs(k1), np.abs(k2))
    smallest_radii = 1.0 / np.abs(min_curvature)
    radii =  tau * smallest_radii

    normals = basemesh.vertex_normals
    inner_v = basemesh.inner_vertices()

    orplanes = np.hstack((normals, -np.sum(normals * V, axis=1)[:, np.newaxis]))

    barycenters  = basemesh.face_barycenters()
    bary_normals = np.mean(normals[basemesh.faces], axis=1)

    # unitize bary_normals
    bary_normals /= np.linalg.norm(bary_normals, axis=1)[:, np.newaxis]
    
    bary_radii = radii[basemesh.faces].mean(axis=1)
    centers = barycenters + bary_radii[:, np.newaxis] * bary_normals

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="dual",
        mode="OrPlanes",
        planes=orplanes,
        radii=radii,
        centers=centers,
        
    )

    experiment_name = f"{name}_tau_{tau:.2f}"
    #lmesh.improve_contact()
    Vs, Fs = lmesh.sphere_centers_mesh()
    lmesh_visualize(lmesh)

    # lmesh.export_data(
    #     output_path, experiment_name )

    opt = setup_optimizer(lmesh)

    for _ in range(10):
        opt.get_gradients()
        opt.optimize_step()
    
        
    # Unpack optimized values
    rf_opt  = opt.unpack("rv")
    npf_opt = opt.unpack("npf").reshape(-1, 3)
    hpf_opt = opt.unpack("hpf")
    cf_opt  = opt.unpack("cv").reshape(-1, 3)
    
    # Update L-mesh with optimized centers (cf), radii (rf), and planes (npf, hpf)
    lmesh.update_spheres_planes(cf_opt, rf_opt, npf_opt, hpf_opt)


    lmesh.improve_contact()

    lmesh_visualize(lmesh) 
        
        # sC, sF = lmesh.sphere_center_mesh()
        # ws.register_surface_mesh("Sphere Centers", sC, sF, color=(0.7, 0.9, 0.7), show_edges=True)

def hl_mesh_computation(name, tau):

    exp_dir = os.path.join(OUT_PATH, name, f"exp_{tau:.2f}")
    os.makedirs(exp_dir, exist_ok=True)

    V, F = read_obj(os.path.join(DATA, name + ".obj"))
    V = normalize_vertices(V, 2)

    print("Reading mesh:", name)

    ref_name = name.replace(".out_d_p", ".obj")

    if ".out_d_p" in name:
        ref_name = name.replace(".out_d_p", ".obj")
    else:
        ref_name = name + ".obj"


    Vr, Fr = read_obj(os.path.join(DATA, ref_name))
    Vr, Fr = ref_surface_mesh(Vr, Fr)
    Vr = normalize_vertices(Vr, 2)

    

    ws.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    # ws.register_surface_mesh("Reference Mesh", Vr, Fr, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    axes, angles = compute_cone_axes(basemesh)
    _, smallest_radii = conical_mesh_smallest_curvature_sphere(
        basemesh.vertices, axes, angles, basemesh.vertex_adjacency_list()
    )

    scaled_axes = axes / np.sin(angles[:, None])

    radii   = tau * smallest_radii

    centers = V + radii[:, np.newaxis] * scaled_axes

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="primal",
        mode="SphCongruence",
        radii=radii,
        centers=centers,
    )

    experiment_name = f"{name}_tau_{tau :.2f}"
    lmesh.export_data(exp_dir, "init")
    
        
    
        # improve_contact() ends with compute_lmesh_structure(), so the improved
        # state is fully rebuilt here and safe to export.
      
    lmesh_visualize(lmesh)

    # lmesh.export_data(
    #     output_path, experiment_name )

    opt = setup_optimizer(lmesh, Vr, Fr)

    for _ in range(10):
        opt.get_gradients()
        opt.optimize_step()
        # Unpack optimized values
    rf_opt  = opt.unpack("rv")
    npf_opt = opt.unpack("npf").reshape(-1, 3)
    hpf_opt = opt.unpack("hpf")
    cf_opt  = opt.unpack("cv").reshape(-1, 3)

    # # Update L-mesh with optimized centers (cf), radii (rf), and planes (npf, hpf)
    lmesh.update_spheres_planes(cf_opt, rf_opt, npf_opt, hpf_opt)

    opt.print_report()
    lmesh.improve_contact()

    lmesh.export_data(exp_dir, "lmesh")
    
    # ws.register_surface_mesh("Gauss L-mesh", onormals, lmesh.top.dual_top(), color=(0.8, 0.8, 0.8), opacity=0.1, show_edges=True, edge_color=(1.0, 0.5, 0.1), edge_radius = 0.002)
    
    lmesh_visualize(lmesh) 



def hl_mesh_simple(name, tau):

    exp_dir = os.path.join(OUT_PATH, name, f"exp_{tau:.2f}")
    os.makedirs(exp_dir, exist_ok=True)

    V, F = read_obj(os.path.join(DATA, name + ".obj"))
    V = normalize_vertices(V, 2)

    print("Reading mesh:", name)

    ref_name = name.replace(".out_d_p", ".obj")

    if ".out_d_p" in name:
        ref_name = name.replace(".out_d_p", ".obj")
    else:
        ref_name = name + ".obj"
    
    ws.register_surface_mesh("Input Mesh", V, F, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    # ws.register_surface_mesh("Reference Mesh", Vr, Fr, color=(0.1, 0.1, 0.1), show_edges=True, edge_color=(1, 1, 1), edge_radius=0.001)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    axes, angles = compute_cone_axes(basemesh)
    _, smallest_radii = conical_mesh_smallest_curvature_sphere(
        basemesh.vertices, axes, angles, basemesh.vertex_adjacency_list()
    )

    scaled_axes = axes / np.sin(angles[:, None])

    radii   = tau * smallest_radii

    centers = V + radii[:, np.newaxis] * scaled_axes

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh,
        orientation=1,
        combinatorics="primal",
        mode="SphCongruence",
        radii=radii,
        centers=centers,
    )

    experiment_name = f"{name}_tau_{tau :.2f}"
    lmesh.export_data(exp_dir, "init")
    
        
    
        # improve_contact() ends with compute_lmesh_structure(), so the improved
        # state is fully rebuilt here and safe to export.
      
    lmesh_visualize(lmesh)

    # lmesh.export_data(
    #     output_path, experiment_name )

    opt = setup_optimizer(lmesh)

    for _ in range(10):
        opt.get_gradients()
        opt.optimize_step()
        # Unpack optimized values
    rf_opt  = opt.unpack("rv")
    npf_opt = opt.unpack("npf").reshape(-1, 3)
    hpf_opt = opt.unpack("hpf")
    cf_opt  = opt.unpack("cv").reshape(-1, 3)

    # # Update L-mesh with optimized centers (cf), radii (rf), and planes (npf, hpf)
    lmesh.update_spheres_planes(cf_opt, rf_opt, npf_opt, hpf_opt)

    opt.print_report()
    lmesh.improve_contact()

    lmesh.export_data(exp_dir, "lmesh")
    
    # ws.register_surface_mesh("Gauss L-mesh", onormals, lmesh.top.dual_top(), color=(0.8, 0.8, 0.8), opacity=0.1, show_edges=True, edge_color=(1.0, 0.5, 0.1), edge_radius = 0.002)
    
    lmesh_visualize(lmesh) 
        

ws.init()   # pings the server; raises if it is not running

ph_mesh_names = [
    "aspiralBdef.out_d_p",
    #"ph_input_cmcOptimized.out_d_p",
    "ph_input_nautilus_ex2.out_d_p",
    #"ph_input_nautilus_channel.out_d_p",
    #"ph_input_seashell_ex5.out_d_p",
    #"ph_input_seashell_ex6.out_d_p",
    #"ph_input_spiral_def.out_d_p",
    #"ph_input_spiral_defM.out_d_p",
    #"TL_mesh_test",
    #"hexMeshTestSP",
    "hexMeshTest"
    #"ph_input_testSimple.out_d_p"
]


ws.clear()
hl_mesh_simple(ph_mesh_names[-1], 0.8)
ws.show()

## Batch sweep — energies, contact deviation, LaTeX tables, view grids

Runs every mesh in `ph_mesh_names` at `tau in [0.5, 0.7, 0.85]`.
Writes one LaTeX table and one image grid per tau under `out/`.

Needs an open viewer tab for the screenshots (`ws.show()`).


In [ ]:
from LMesh.lmesh_serializers import lmesh_visualize
# ── Batch sweep: every mesh x every tau, with LaTeX tables and view grids ──────
#
# For each (mesh, tau):
#   * build the L-mesh, run 10 optimisation steps
#   * record the final optimisation energy (total and per term)
#   * run improve_contact() and record the oriented contact deviation
#   * record the weights actually used
#   * capture two views of the scene
#
# Outputs, one set per tau value:
#   out/tables/ph_energy_tau[TAU].tex        -- LaTeX table, ready to \input
#   out/images/ph_images/grid_tau[TAU].png   -- one row per mesh, two views
#   out/images/ph_images/[mesh]_[tau].png    -- the individual views

import os, json, io, re, contextlib
import numpy as np

TAUS       = [0.5, 0.7, 0.85]
N_STEPS    = 10

# ── Framing for the grid images ───────────────────────────────────────────────
# Two fixed three-quarter angles, each mesh fitted to the frame: every row is
# sized consistently and a re-run reproduces the same pictures.
#
# ZOOM is the only knob you normally touch -- it scales the auto-fit distance,
# so LOWER means CLOSER and less empty background:
#     1.05  default fit, generous margin
#     0.85  moderately tight
#     0.75  tight   <- current
#     0.65  very tight; a long/curved mesh may clip at the edges
ZOOM = 0.65

VIEWS = [dict(v, zoom=ZOOM) for v in ws.TIGHT_VIEWS]

# To use different angles, edit `dir` here (up is +y):
#   VIEWS = [{"dir": (1, 0.7, 1), "up": (0, 1, 0), "zoom": ZOOM},
#            {"dir": (-1, 0.5, -0.8), "up": (0, 1, 0), "zoom": ZOOM}]
# Or capture with your live viewer camera instead:  VIEWS = ws.DEFAULT_VIEWS
IMG_DIR    = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "images", "ph_images")
TAB_DIR    = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "tables")
# L-mesh exports land in notebooks/out/HL_results/[name]/exp_[tau]/, with the
# initial and optimised states side by side under that folder.
EXPORT_DIR = os.path.join(PROJECT_ROOT, "notebooks", "out", "lmesh", "HL_results")
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

# These meshes are stored with outward-pointing normals while the rest point
# inward; reversing the face winding flips them so every input agrees.
# (Checked via signed volume: positive => outward under this winding.)
def _signed_volume(V, faces):
    vol = 0.0
    for f in faces:
        f = list(f)
        for k in range(1, len(f) - 1):
            vol += np.dot(V[f[0]], np.cross(V[f[k]], V[f[k + 1]])) / 6.0
    return vol


def load_oriented(name, data_dir):
    """Read a mesh and return (V, F) with consistently inward-pointing normals."""
    V, F = read_obj(os.path.join(data_dir, name + ".obj"))
    V = normalize_vertices(V, 2)
    F = [list(f) for f in F]
    if _signed_volume(V, F) > 0:
        F = [f[::-1] for f in F]     # reverse winding -> flip normals
        print(f"    [orient] flipped normals for {name}")
    return V, np.array(F), F


def run_case(name, tau, data_dir=None):
    """Build + optimise one (mesh, tau) case.  Returns a record dict."""
    data_dir = data_dir or DATA
    V, F, Flist = load_oriented(name, data_dir)

    basemesh = Mesh()
    basemesh.make_mesh(V, F)

    axes, angles = compute_cone_axes(basemesh)
    _, smallest_radii = conical_mesh_smallest_curvature_sphere(
        basemesh.vertices, axes, angles, basemesh.vertex_adjacency_list()
    )
    scaled_axes = axes / np.sin(angles[:, None])
    radii   = tau * smallest_radii
    centers = basemesh.vertices + radii[:, None] * scaled_axes

    lmesh = Lmesh()
    lmesh.init_L_mesh(
        basemesh, orientation=1, combinatorics="primal",
        mode="SphCongruence", radii=radii, centers=centers,
    )

    # notebooks/out/HL_results/[name]/exp_[tau]/
    exp_dir = os.path.join(EXPORT_DIR, name, f"exp_{tau:.2f}")
    os.makedirs(exp_dir, exist_ok=True)

    # init_L_mesh() already ran compute_lmesh_structure(), so the initial state
    # is complete and exportable before any optimisation touches it.
    with contextlib.redirect_stdout(io.StringIO()):
        lmesh.export_data(exp_dir, "init")

    opt = setup_optimizer(lmesh, *ref_pair(name, data_dir))

    for _ in range(N_STEPS):
        opt.get_gradients()
        opt.optimize_step()

    # Final total optimisation energy (weighted, as the optimiser sees it)
    X = opt.X
    total_e = float(sum(t.compute_energy(X) for t in opt.objective_terms.values()))

    lmesh.update_spheres_planes(
        opt.unpack("cv").reshape(-1, 3), opt.unpack("rv"),
        opt.unpack("npf").reshape(-1, 3), opt.unpack("hpf"),
    )

    # Oriented contact deviation of the optimised L-mesh.
    lmesh.improve_contact()
    dev = lmesh.compute_oriented_contact_dev()

    # improve_contact() ends with compute_lmesh_structure(), so the optimised
    # state is fully rebuilt here and safe to export.
    with contextlib.redirect_stdout(io.StringIO()):
        lmesh.export_data(exp_dir, "lmesh")
    print(f"    exported -> {os.path.relpath(exp_dir, PROJECT_ROOT)}")

    n_contact = len(lmesh.get_contact_points())

    return {
        "mesh": name, "tau": tau, "export_dir": exp_dir,
        "V": int(lmesh.top.V), "F": int(lmesh.top.F),
        "energy_total": total_e, "contact_dev": dev,
        "n_contact_points": n_contact,
        "lmesh": lmesh,
    }


_REF_CACHE = {}

def ref_pair(name, data_dir):
    """Remeshed reference surface for the proximity/glide terms.

    setup_optimizer registers ContactPointsProximity / ContactPointsGlide
    unconditionally (at weight 0), and both need a real (Vr, Fr), so build the
    same reference hl_mesh_computation uses.  Cached: it is the same surface for
    every tau of a given mesh.
    """
    if name in _REF_CACHE:
        return _REF_CACHE[name]
    ref_name = name.replace(".out_d_p", "")
    Vr, Fr = read_obj(os.path.join(data_dir, ref_name + ".obj"))
    Vr, Fr = ref_surface_mesh(Vr, Fr)
    Vr = normalize_vertices(Vr, 2)
    _REF_CACHE[name] = (Vr, Fr)
    return _REF_CACHE[name]


def latex_table(records, tau):
    """Render one tau's records as a LaTeX table (booktabs)."""
    def esc(s):
        return str(s).replace("_", r"\_")

    # Columns: mesh name (leftmost), |V|, |F|, total energy, contact deviation.
    lines = []
    lines.append("% Generated by hl_mesh_lab.ipynb -- tau = " + ("%.2f" % tau))
    lines.append(r"% Requires: \usepackage{booktabs}")
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"  \centering")
    lines.append(r"  \small")
    lines.append(r"  \begin{tabular}{lrrrr}")
    lines.append(r"    \toprule")
    head = ["Mesh", r"$|V|$", r"$|F|$", r"$E_{\mathrm{total}}$",
            r"$\delta_{\mathrm{contact}}$"]
    lines.append("    " + " & ".join(head) + r" \\")
    lines.append(r"    \midrule")
    for r in records:
        row = [esc(r["mesh"]), str(r["V"]), str(r["F"]),
               "%.3e" % r["energy_total"], "%.3e" % r["contact_dev"]]
        lines.append("    " + " & ".join(row) + r" \\")
    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append("  \\caption{HL-mesh optimisation after " + str(N_STEPS) +
                 " steps, $\\tau = " + ("%.2f" % tau) + "$. "
                 "$\\delta_{\\mathrm{contact}}$ is the oriented contact "
                 "deviation after \\texttt{improve\\_contact()}.}")
    lines.append("  \\label{tab:hlmesh-tau-" + str(tau).replace(".", "p") + "}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


# ── Sweep ─────────────────────────────────────────────────────────────────────
all_records = {}

for tau in TAUS:
    print(f"\n{'='*70}\n  tau = {tau}\n{'='*70}")
    records, grid_rows, grid_labels = [], [], []

    for name in ph_mesh_names:
        print(f"  [{name}]  tau={tau}")
        try:
            rec = run_case(name, tau)
        except Exception as e:
            print(f"    SKIP {type(e).__name__}: {e}")
            continue

        # Draw this case alone, then capture the two views.
        ws.clear()
        lmesh_visualize(rec["lmesh"])
        try:
            blobs = ws.screenshot_views(views=VIEWS, width=640, height=480)
            for i, b in enumerate(blobs):
                fn = os.path.join(IMG_DIR, f"{name}_{tau:.2f}_v{i}.png")
                with open(fn, "wb") as fh:
                    fh.write(b)
            grid_rows.append(blobs)
            grid_labels.append(f"{name}   tau={tau:.2f}   "
                               f"E={rec['energy_total']:.2e}   "
                               f"dev={rec['contact_dev']:.2e}")
        except Exception as e:
            print(f"    (no screenshot: {e})")

        rec.pop("lmesh", None)          # don't keep meshes alive in the record
        records.append(rec)
        print(f"    E_total={rec['energy_total']:.4e}   "
              f"contact_dev={rec['contact_dev']:.4e}   "
              f"contact_pts={rec['n_contact_points']}")

    all_records[tau] = records

    # LaTeX table for this tau
    if records:
        tex  = latex_table(records, tau)
        path = os.path.join(TAB_DIR, f"ph_energy_tau{tau:.2f}.tex")
        with open(path, "w") as fh:
            fh.write(tex + "\n")
        print(f"\n  -> wrote {path}")
        print(tex)

    # Contact-sheet grid for this tau
    if grid_rows:
        gpath = os.path.join(IMG_DIR, f"grid_tau{tau:.2f}.png")
        ws.save_grid(gpath, grid_rows, labels=grid_labels)
        print(f"  -> wrote {gpath}")

# Raw numbers, in case you want to re-render the tables differently
with open(os.path.join(TAB_DIR, "ph_sweep_results.json"), "w") as fh:
    json.dump(all_records, fh, indent=2)
print(f"\nDone. Tables in {TAB_DIR}, images in {IMG_DIR}")
